# Three-dimensional Wave equation example

This notebook runs the existing Weak-PDE-Net Wave3D pipeline and records representative text results after Searching, Pruning, and Tuning. No visualization is generated or displayed.

All model and training hyperparameters are loaded directly from `Wave_3d_mains/wave3d_params.py`. Only the sampling ratio, noise level, and optional number of sampling points are selected below.

> To retain all outputs in this file when running from the command line, execute the notebook with `jupyter nbconvert --execute --to notebook --inplace 3D_example.ipynb`.

In [1]:
from contextlib import redirect_stderr, redirect_stdout
from pathlib import Path
import os
import sys

import numpy as np
import torch

# Locate the repository whether Jupyter starts here or in a subdirectory.
current_dir = Path.cwd().resolve()
project_candidates = [current_dir, *current_dir.parents]
PROJECT_ROOT = next(
    (path for path in project_candidates if (path / 'PDE_Discover.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Run this notebook from the repository or Wave_3d_mains directory.'
    )

WAVE3D_DIR = PROJECT_ROOT / 'Wave_3d_mains'
for path in (PROJECT_ROOT, PROJECT_ROOT / 'datasets', WAVE3D_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
os.chdir(WAVE3D_DIR)

from wave3d import (
    ensure_dataset,
    get_incremental_sampling_idx_by_count,
    load_wave3d_data,
    set_all_seeds,
)
from wave3d_params import Params
from PDE_Discover import PDE_Discover
from Tune_Net import evaluate_test_mse

## 1. Reproduction settings

The default sample count comes from `wave3d_params.py`. Set `SAMPLE_POINTS` to `None` to use `SAMPLE_RATIO` instead; all remaining parameters are unchanged.

In [2]:
config = Params()
default_counts = list(getattr(config, 'default_sample_counts', []))
SAMPLE_RATIO = float(getattr(config, 'sample_ratio', 0.006))
NOISE_LEVEL = float(getattr(config, 'sigma_NR', 0.0))
SAMPLE_POINTS = int(default_counts[1 if len(default_counts) > 1 else 0]) if default_counts else None

SEED = int(getattr(config, 'seed', 42))
set_all_seeds(SEED)

## 2. Load and sample the Wave3D data

In [3]:
data_path = ensure_dataset(PROJECT_ROOT / config.data_file)
coordinates, values, full_data, metadata = load_wave3d_data(data_path)
total_points = int(metadata['total_points'])

if SAMPLE_POINTS is None:
    if not 0 < SAMPLE_RATIO <= 1:
        raise ValueError('SAMPLE_RATIO must be in the interval (0, 1].')
    sample_points = int(total_points * SAMPLE_RATIO)
else:
    if not 0 < SAMPLE_POINTS <= total_points:
        raise ValueError(f'SAMPLE_POINTS must be between 1 and {total_points}.')
    sample_points = int(SAMPLE_POINTS)
sample_ratio = sample_points / total_points

config.sample_count = sample_points
config.sample_ratio = sample_ratio
config.sigma_NR = NOISE_LEVEL
config.grid_size = list(metadata['grid_size'])
config.grid_plot_size = list(metadata['grid_size'])
config.input_bounds = metadata['input_bounds']
config.run_tag = f'N{sample_points}'

with open(os.devnull, 'w') as quiet_output, redirect_stdout(quiet_output):
    sampling_idx = get_incremental_sampling_idx_by_count(
        total_size=total_points,
        target_count=sample_points,
        save_dir=WAVE3D_DIR / 'sampling_idx' / 'reviewer_demo',
        seed=SEED,
    )

coordinates_train = torch.tensor(
    coordinates[sampling_idx], dtype=torch.float32, requires_grad=True
)
values_train = torch.tensor(values[sampling_idx], dtype=torch.float32)
if NOISE_LEVEL > 0:
    noise_scale = NOISE_LEVEL * torch.sqrt(torch.mean(values_train ** 2))
    values_train = values_train + torch.randn_like(values_train) * noise_scale
input_train = torch.cat((coordinates_train, values_train), dim=1)
input_data = torch.tensor(full_data, dtype=torch.float32)

print(f'Dataset: {data_path.name}')
print(f'Grid shape: {tuple(config.grid_size)}')
print(f'Training samples: {sample_points} ({sample_ratio:.6%})')
print(f'Noise level: {NOISE_LEVEL}')
print('Expected PDE: D_tt(u) = D_x^2(u) + D_y^2(u) + D_z^2(u)')

Dataset: Wave3D_Analytic.npz
Grid shape: (50, 32, 32, 32)
Training samples: 10000 (0.610352%)
Noise level: 0.0
Expected PDE: D_tt(u) = D_x^2(u) + D_y^2(u) + D_z^2(u)


## 3. Run the discovery pipeline

The complete Searching, Pruning, and Tuning workflow is executed without changing its internal logic. Verbose epoch logs are hidden, while the representative text results and project test loss from each stage are retained.

In [4]:
stage_results = {}
with open(os.devnull, 'w') as quiet_output:
    with redirect_stdout(quiet_output), redirect_stderr(quiet_output):
        solver_input = input_train if config.data_spase else input_data
        solver = PDE_Discover(config, input_data=solver_input, inputs_test=input_data)

def project_test_loss(current_solver):
    return evaluate_test_mse(
        current_solver.Pig_Net,
        current_solver.inputs_test,
        current_solver.problem_dim,
        current_solver.Device,
        current_solver.norm_flag,
    )

original_nas = solver.NAS_Sym_Net
def tracked_nas():
    result = original_nas()
    beta_list, alphas_list, _, _, _ = result
    operator_names = [operator.__class__.__name__ for operator in solver.funcs]
    selected_architectures = []
    for equation_index, (beta, equation_alphas) in enumerate(
        zip(beta_list, alphas_list)
    ):
        depth = solver.depth_candidates[int(np.argmax(np.asarray(beta)))]
        layers = []
        for layer_alphas in equation_alphas[:depth]:
            repeats = {}
            for operator_index, (name, alpha) in enumerate(
                zip(operator_names, layer_alphas)
            ):
                candidate_index = int(np.argmax(np.asarray(alpha)))
                repeats[name] = int(
                    solver.repeats_candidates[operator_index][candidate_index]
                )
            layers.append(repeats)
        selected_architectures.append(
            {'equation': equation_index + 1, 'depth': depth, 'layers': layers}
        )
    stage_results['Searching'] = {
        'architectures': selected_architectures,
        'test_loss': project_test_loss(solver),
    }
    return result
solver.NAS_Sym_Net = tracked_nas

original_final_train = solver.final_train
def tracked_final_train():
    result = original_final_train()
    sparse_equations = []
    for equation_index, payload in sorted(result.items()):
        loss = payload['total_loss']
        if torch.is_tensor(loss):
            loss = loss.detach().cpu().item()
        sparse_equations.append({
            'equation': equation_index + 1,
            'pde': str(np.asarray(payload['pde'], dtype=object).reshape(-1)[0]),
            'total_loss': float(loss),
        })
    stage_results['Pruning'] = {
        'equations': sparse_equations,
        'test_loss': project_test_loss(solver),
    }
    return result
solver.final_train = tracked_final_train

with open(os.devnull, 'w') as quiet_output:
    with redirect_stdout(quiet_output), redirect_stderr(quiet_output):
        final_pdes = solver.Solve_problem()

stage_results['Tuning'] = {
    'enabled': config.n_epochs_tune > 0,
    'equations': final_pdes,
    'test_loss': project_test_loss(solver),
}

print('Searching completed:')
for architecture in stage_results['Searching']['architectures']:
    print(f"  Equation {architecture['equation']}: depth = {architecture['depth']}")
    for layer_index, repeats in enumerate(architecture['layers'], start=1):
        selected = ', '.join(
            f'{name} x {count}' for name, count in repeats.items() if count > 0
        ) or 'none'
        print(f'    Layer {layer_index}: {selected}')
print(f"  Test loss: {stage_results['Searching']['test_loss']:.3e}")

print('\nPruning completed:')
for result in stage_results['Pruning']['equations']:
    print(f"  Equation {result['equation']}: {result['pde']}")
    print(f"  Total loss: {result['total_loss']:.3e}")
print(f"  Test loss: {stage_results['Pruning']['test_loss']:.3e}")

if stage_results['Tuning']['enabled']:
    print('\nTuning completed:')
else:
    print('\nTuning skipped; final post-pruning equation:')
for pde in stage_results['Tuning']['equations']:
    print(f'  {pde}')
print(f"  Test loss: {stage_results['Tuning']['test_loss']:.3e}")

Searching completed:
  Equation 1: depth = 2
    Layer 1: Identity x 3, Square x 1
    Layer 2: Identity x 1, Product x 1
  Test loss: 2.643e+00

Pruning completed:
  Equation 1: -0.33177*(-0.0309801*u**2 + 0.487916*u - 0.258052289485931*(-0.0676646*u**2 + 0.32837*u)*(0.0605443*u**2 - 0.300289*u)) + 0.107259*D_x (0.056104*u**2 - 0.8836*u - 0.755315065383911*(-0.0676646*u**2 + 0.32837*u)*(0.0605443*u**2 - 0.300289*u)) - 0.245151*D_x D_y (0.0518002*u**2 - 0.815817*u + 0.0861772149801254*(-0.0676646*u**2 + 0.32837*u)*(0.0605443*u**2 - 0.300289*u)) - 0.264932*D_x D_z (0.0556996*u**2 - 0.877231*u + 0.148661375045776*(-0.0676646*u**2 + 0.32837*u)*(0.0605443*u**2 - 0.300289*u)) + 0.943353*D_x^2 (-0.0627044*u**2 + 0.987551*u - 0.461892277002335*(-0.0676646*u**2 + 0.32837*u)*(0.0605443*u**2 - 0.300289*u)) + 0.538776*D_y (0.00301621*u**2 - 0.0475033*u - 0.835883259773254*(-0.0676646*u**2 + 0.32837*u)*(0.0605443*u**2 - 0.300289*u)) - 0.0263525*D_y D_z (0.164944589138031*(-0.0676646*u**2 + 0.32837

## 4. Final text result

In [5]:
print('Expected PDE:')
print('  D_tt(u) = D_x^2(u) + D_y^2(u) + D_z^2(u)')
print('Discovered PDE:')
for pde in final_pdes:
    print(f'  {pde}')
print(f'Final test loss: {stage_results["Tuning"]["test_loss"]:.3e}')

Expected PDE:
  D_tt(u) = D_x^2(u) + D_y^2(u) + D_z^2(u)
Discovered PDE:
  D_tt(u)=0.9987162002688073*D_z^2 (u)+0.9878954516977277*D_y^2 (u)+0.9645130740374677*D_x^2 (u)
Final test loss: 2.712e+00
